# 11 — Visualizaciones globales para memoria y presentación

Este notebook consolida visualizaciones globales del pipeline hasta la construcción del **proxy ground truth textual**.

**Importante:** no recalcula diarización, embeddings, transcripciones ni alineaciones. Solo lee outputs generados por notebooks anteriores y produce tablas/figuras limpias para la memoria, la presentación y la defensa.

Fases cubiertas:

1. Inventario y limpieza acústica.
2. Diarización, anchors y reetiquetado.
3. Consolidación de segmentos.
4. Transcripción automática segmentada.
5. Metadata oficial, alineación textual y proxy ground truth.

In [ ]:
# ============================================================
# CELDA 1 - IMPORTS
# ============================================================

from pathlib import Path
from pandas.errors import EmptyDataError
from IPython.display import display, Markdown

import re
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# CELDA 2 - CONFIGURACIÓN DE RUTAS Y ESTILO VISUAL
# ============================================================

# Ruta base del proyecto en la VM
PROJECT_DIR = Path("/home/jupyter/TFM_ProcesadoDeAudios")
DATA_DIR = PROJECT_DIR / "data"

# Entradas por fase
EDA_DIR = DATA_DIR / "eda"
DIARIZATION_DIR = DATA_DIR / "diarization_outputs"
FINAL_RELABEL_DIR = DIARIZATION_DIR / "final_relabel"
CONSOLIDATED_DIR = DIARIZATION_DIR / "consolidated"
TRANSCRIPTION_DIR = DATA_DIR / "transcription_outputs"
PROXY_DIR = DATA_DIR / "proxy_groundtruth_outputs"

# Salidas de este notebook
NOTEBOOK11_OUTPUT_DIR = DATA_DIR / "global_visualizations_memory_presentation"
FIGURES_DIR = NOTEBOOK11_OUTPUT_DIR / "figures"
TABLES_DIR = NOTEBOOK11_OUTPUT_DIR / "tables"

for d in [NOTEBOOK11_OUTPUT_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Configuración visual común
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# Paleta sencilla y consistente para las figuras de memoria/presentación
PALETTE = [
    "#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2",
    "#B279A2", "#FF9DA6", "#9D755D", "#BAB0AC"
]

print("PROJECT_DIR:", PROJECT_DIR)
print("Salidas Notebook 11:", NOTEBOOK11_OUTPUT_DIR)
print("Figuras:", FIGURES_DIR)
print("Tablas:", TABLES_DIR)

In [ ]:
# ============================================================
# CELDA 3 - FUNCIONES AUXILIARES
# ============================================================

def first_existing_path(paths):
    """Devuelve el primer path existente; si ninguno existe, devuelve el primero para diagnóstico."""
    paths = [Path(p) for p in paths]
    for p in paths:
        if p.exists():
            return p
    return paths[0] if paths else None


def read_csv_optional(path, **kwargs):
    """Lee un CSV si existe. Si no existe o está vacío, devuelve DataFrame vacío."""
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path, **kwargs)
    except EmptyDataError:
        return pd.DataFrame()
    except Exception as e:
        print(f"No se pudo leer {path.name}: {e}")
        return pd.DataFrame()


def save_table(df, filename):
    """Guarda una tabla en TABLES_DIR y devuelve la ruta."""
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    print("Tabla guardada:", path)
    return path


def save_fig(filename):
    """Guarda la figura actual en FIGURES_DIR."""
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print("Figura guardada:", path)
    return path


def normalize_audio_id(x):
    """Normaliza identificadores de audio para agrupaciones simples."""
    if pd.isna(x):
        return np.nan
    s = Path(str(x).strip()).name
    s = re.sub(r"\.csv$", "", s)
    s = re.sub(r"\.wav$", "", s)
    s = re.sub(r"_final_merged$", "", s)
    s = re.sub(r"_transcribed_segments$", "", s)
    return s


def find_first_col(df, candidates):
    """Devuelve la primera columna existente de una lista de candidatas."""
    for c in candidates:
        if c in df.columns:
            return c
    return None


def detect_audio_col(df):
    return find_first_col(df, [
        "audio_file", "audio_base", "audio_stem", "file_stem", "audio_id",
        "filename", "file", "source_file", "clean_audio_file"
    ])


def detect_text_col(df):
    return find_first_col(df, [
        "text", "transcription", "transcribed_text", "whisper_text",
        "segment_text", "texto", "transcripcion_whisper"
    ])


def detect_start_end_cols(df):
    start_col = find_first_col(df, ["start", "start_sec", "start_time", "segment_start"])
    end_col = find_first_col(df, ["end", "end_sec", "end_time", "segment_end"])
    return start_col, end_col


def format_seconds(x):
    """Convierte segundos a mm:ss.s para tablas/ejemplos."""
    try:
        x = float(x)
    except Exception:
        return ""
    m = int(x // 60)
    s = x - 60 * m
    return f"{m:02d}:{s:04.1f}"


def add_interval_column(df, start_col=None, end_col=None, out_col="interval"):
    df = df.copy()
    if start_col is None or end_col is None:
        start_col, end_col = detect_start_end_cols(df)
    if start_col and end_col and start_col in df.columns and end_col in df.columns:
        df[out_col] = df[start_col].apply(format_seconds) + " - " + df[end_col].apply(format_seconds)
    return df


def plot_bar_counts(counts, title, xlabel, ylabel, filename, horizontal=False, value_labels=True):
    """Grafica una serie de conteos."""
    counts = counts.dropna()
    if len(counts) == 0:
        print("Sin datos para:", title)
        return None
    fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(counts))) if horizontal else (9, 5))
    colors = PALETTE[:len(counts)]
    if horizontal:
        ax.barh(counts.index.astype(str), counts.values, color=colors)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.invert_yaxis()
        if value_labels:
            for i, v in enumerate(counts.values):
                ax.text(v, i, f" {int(v):,}".replace(",", "."), va="center")
    else:
        ax.bar(counts.index.astype(str), counts.values, color=colors)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=30)
        if value_labels:
            for i, v in enumerate(counts.values):
                ax.text(i, v, f"{int(v):,}".replace(",", "."), ha="center", va="bottom")
    ax.set_title(title)
    return save_fig(filename)


def plot_hist(series, title, xlabel, filename, bins=30, clip_quantile=None):
    """Histograma robusto con opción de recorte por cuantiles para evitar outliers extremos."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if clip_quantile is not None and len(s) > 0:
        upper = s.quantile(clip_quantile)
        s = s[s <= upper]
    if len(s) == 0:
        print("Sin datos para:", title)
        return None
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(s, bins=bins, color=PALETTE[0], alpha=0.85, edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Frecuencia")
    return save_fig(filename)


def plot_segment_funnel(stage_rows, filename="pipeline_segment_funnel.png"):
    """Gráfico horizontal de reducción/transformación de segmentos por fase."""
    df = pd.DataFrame(stage_rows).dropna(subset=["value"])
    df = df[df["value"] > 0].copy()
    if df.empty:
        print("No hay etapas suficientes para el funnel.")
        return pd.DataFrame()
    df["value"] = df["value"].astype(int)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(df))))
    ax.barh(df["stage"], df["value"], color=PALETTE[0])
    ax.invert_yaxis()
    ax.set_title("Evolución global de segmentos por fase del pipeline")
    ax.set_xlabel("Número de segmentos")
    ax.set_ylabel("Fase")
    for i, v in enumerate(df["value"]):
        ax.text(v, i, f" {v:,}".replace(",", "."), va="center")
    save_fig(filename)
    return df


def plot_crosstab_heatmap(ct, title, filename):
    """Heatmap simple con matplotlib, sin seaborn."""
    if ct.empty:
        print("Sin datos para:", title)
        return None
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(ct.values, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(ct.columns)))
    ax.set_yticks(np.arange(len(ct.index)))
    ax.set_xticklabels(ct.columns)
    ax.set_yticklabels(ct.index)
    ax.set_xlabel("Etiqueta final")
    ax.set_ylabel("Etiqueta original")
    for i in range(ct.shape[0]):
        for j in range(ct.shape[1]):
            ax.text(j, i, f"{int(ct.iloc[i, j]):,}".replace(",", "."),
                    ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return save_fig(filename)


def safe_value_count(df, col):
    if df.empty or col not in df.columns:
        return pd.Series(dtype=int)
    return df[col].value_counts(dropna=False)


def compute_word_count_from_text(df, text_col):
    if df.empty or text_col is None:
        return pd.Series(dtype=float)
    return (
        df[text_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .apply(lambda x: len(x.split()) if x else 0)
    )

In [ ]:
# ============================================================
# CELDA 4 - REGISTRO DE ARCHIVOS ESPERADOS
# ============================================================

CSV_PATHS = {
    # EDA / limpieza
    "audio_inventory_private": EDA_DIR / "audio_inventory_private.csv",
    "audio_inventory_integrated": EDA_DIR / "audio_inventory_integrated.csv",
    "audio_cleaning_summary": EDA_DIR / "audio_cleaning_summary.csv",

    # Diarización / relabeling
    "diarization_summary": DIARIZATION_DIR / "diarization_summary.csv",
    "regular_segments": DIARIZATION_DIR / "diarization_all_regular_segments.csv",
    "scored_segments": DIARIZATION_DIR / "diarization_all_scored_segments.csv",
    "valid_segments": DIARIZATION_DIR / "diarization_all_valid_segments.csv",
    "anchor_segments": DIARIZATION_DIR / "diarization_all_anchor_segments.csv",
    "relabel_summary": FINAL_RELABEL_DIR / "relabel_summary.csv",
    "final_segments": FINAL_RELABEL_DIR / "all_final_segments.csv",
    "changed_segments": FINAL_RELABEL_DIR / "all_changed_segments.csv",
    "final_merged_segments": first_existing_path([
        CONSOLIDATED_DIR / "all_final_merged_segments.csv",
        FINAL_RELABEL_DIR / "all_final_merged_segments.csv",
    ]),

    # Validación/sensibilidad del Notebook 02
    "relabel_margin_sensitivity": DIARIZATION_DIR / "relabel_margin_sensitivity" / "relabel_margin_sensitivity_summary.csv",
    "anchor_overlap_sensitivity": DIARIZATION_DIR / "overlap_anchor_sensitivity" / "anchor_overlap_threshold_sensitivity.csv",

    # Transcripción segmentada
    "transcription_summary": TRANSCRIPTION_DIR / "transcription_summary.csv",
    "transcribed_segments": first_existing_path([
        TRANSCRIPTION_DIR / "06_transcribed_segments_final.csv",
        TRANSCRIPTION_DIR / "all_segments_transcribed.csv",
    ]),

    # Metadata oficial y proxy ground truth textual
    "metadata_join_audit_by_audio": PROXY_DIR / "metadata_join_audit_by_audio.csv",
    "official_role_presence_by_audio": PROXY_DIR / "official_role_presence_by_audio.csv",
    "metadata_role_summary": PROXY_DIR / "metadata_role_summary.csv",
    "official_transcription_turns_extracted": PROXY_DIR / "official_transcription_turns_extracted.csv",
    "text_alignment_candidates": PROXY_DIR / "text_alignment_candidates.csv",
    "text_alignment_matches": PROXY_DIR / "text_alignment_matches.csv",
    "text_alignment_threshold_sensitivity": PROXY_DIR / "text_alignment_threshold_sensitivity.csv",
    "speaker_role_mapping_textual": PROXY_DIR / "speaker_role_mapping_textual.csv",
    "segment_level_proxy_groundtruth": PROXY_DIR / "segment_level_proxy_groundtruth.csv",
    "textual_proxy_metrics_summary": PROXY_DIR / "textual_proxy_metrics_summary.csv",
    "alignment_processing_summary": PROXY_DIR / "alignment_processing_summary.csv",
    "holdout_role_mapping_predictions": PROXY_DIR / "holdout_role_mapping_predictions.csv",
    "holdout_role_mapping_metrics": PROXY_DIR / "holdout_role_mapping_metrics.csv",
}

path_status = pd.DataFrame([
    {
        "name": name,
        "path": str(path),
        "exists": Path(path).exists() if path is not None else False,
        "size_mb": round(Path(path).stat().st_size / 1024**2, 3) if path is not None and Path(path).exists() else np.nan,
    }
    for name, path in CSV_PATHS.items()
])

save_table(path_status, "00_path_status.csv")
display(path_status)

In [ ]:
# ============================================================
# CELDA 5 - CARGA DE OUTPUTS EXISTENTES
# ============================================================

dfs = {name: read_csv_optional(path) for name, path in CSV_PATHS.items()}

loaded_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "loaded": len(df) > 0,
    }
    for name, df in dfs.items()
])

save_table(loaded_summary, "01_loaded_datasets_summary.csv")
display(loaded_summary)

## 1. Resumen global del pipeline

Esta sección genera una tabla única de métricas globales y un funnel de segmentos por fase. Es útil para explicar el recorrido del corpus desde la diarización hasta el proxy textual.

In [ ]:
# ============================================================
# CELDA 6 - KPIs GLOBALES DEL PIPELINE
# ============================================================

kpi_rows = []


def add_kpi(phase, metric, value, notes=""):
    if value is None:
        return
    try:
        if pd.isna(value):
            return
    except Exception:
        pass
    kpi_rows.append({
        "phase": phase,
        "metric": metric,
        "value": value,
        "notes": notes,
    })

# Inventario / limpieza
for inv_name in ["audio_inventory_private", "audio_inventory_integrated"]:
    df_inv = dfs.get(inv_name, pd.DataFrame())
    if len(df_inv):
        add_kpi("00_EDA_limpieza", "Audios en inventario", int(len(df_inv)), inv_name)
        break

# Diarización
add_kpi("01_diarizacion", "Audios con resumen de diarización", int(len(dfs["diarization_summary"])) if len(dfs["diarization_summary"]) else np.nan)
add_kpi("01_diarizacion", "Segmentos regulares", int(len(dfs["regular_segments"])) if len(dfs["regular_segments"]) else np.nan)
add_kpi("01_diarizacion", "Segmentos puntuados", int(len(dfs["scored_segments"])) if len(dfs["scored_segments"]) else np.nan)
add_kpi("01_diarizacion", "Segmentos válidos", int(len(dfs["valid_segments"])) if len(dfs["valid_segments"]) else np.nan)
add_kpi("01_diarizacion", "Anchors seleccionados", int(len(dfs["anchor_segments"])) if len(dfs["anchor_segments"]) else np.nan)

# Relabeling
if len(dfs["relabel_summary"]):
    status_col = "status" if "status" in dfs["relabel_summary"].columns else None
    if status_col:
        ok_mask = dfs["relabel_summary"][status_col].astype(str).str.lower().isin(["ok", "correcto", "completed", "success"])
        add_kpi("01_relabeling", "Audios con relabeling OK", int(ok_mask.sum()))
        add_kpi("01_relabeling", "Audios omitidos/no OK", int((~ok_mask).sum()))
    else:
        add_kpi("01_relabeling", "Audios en resumen de relabeling", int(len(dfs["relabel_summary"])))

add_kpi("01_relabeling", "Segmentos finales post-relabeling", int(len(dfs["final_segments"])) if len(dfs["final_segments"]) else np.nan)
add_kpi("01_relabeling", "Segmentos reetiquetados", int(len(dfs["changed_segments"])) if len(dfs["changed_segments"]) else np.nan)

# Consolidación
add_kpi("04_consolidacion", "Segmentos finales fusionados", int(len(dfs["final_merged_segments"])) if len(dfs["final_merged_segments"]) else np.nan)

# Transcripción
add_kpi("05_transcripcion", "Segmentos transcritos", int(len(dfs["transcribed_segments"])) if len(dfs["transcribed_segments"]) else np.nan)
if len(dfs["transcribed_segments"]):
    text_col = detect_text_col(dfs["transcribed_segments"])
    if text_col:
        n_with_text = int(dfs["transcribed_segments"][text_col].fillna("").astype(str).str.strip().ne("").sum())
        add_kpi("05_transcripcion", "Segmentos con texto Whisper", n_with_text)
        add_kpi("05_transcripcion", "Segmentos sin texto Whisper", int(len(dfs["transcribed_segments"]) - n_with_text))

# Proxy textual
add_kpi("06_proxy_groundtruth", "Turnos oficiales extraídos", int(len(dfs["official_transcription_turns_extracted"])) if len(dfs["official_transcription_turns_extracted"]) else np.nan)
add_kpi("06_proxy_groundtruth", "Candidatos de alineación textual", int(len(dfs["text_alignment_candidates"])) if len(dfs["text_alignment_candidates"]) else np.nan)
add_kpi("06_proxy_groundtruth", "Matches textuales aceptados", int(len(dfs["text_alignment_matches"])) if len(dfs["text_alignment_matches"]) else np.nan)
add_kpi("06_proxy_groundtruth", "Mappings speaker→rol", int(len(dfs["speaker_role_mapping_textual"])) if len(dfs["speaker_role_mapping_textual"]) else np.nan)

if len(dfs["segment_level_proxy_groundtruth"]):
    role_col = find_first_col(dfs["segment_level_proxy_groundtruth"], ["official_role_proxy", "proxy_role", "probable_role", "role_proxy"])
    if role_col:
        n_proxy = int(dfs["segment_level_proxy_groundtruth"][role_col].notna().sum())
        add_kpi("06_proxy_groundtruth", "Segmentos etiquetados con proxy textual", n_proxy, role_col)
    else:
        add_kpi("06_proxy_groundtruth", "Segmentos en salida proxy", int(len(dfs["segment_level_proxy_groundtruth"])))

kpi_df = pd.DataFrame(kpi_rows)
save_table(kpi_df, "02_pipeline_global_kpis.csv")
display(kpi_df)

In [ ]:
# ============================================================
# CELDA 7 - FUNNEL GLOBAL DE SEGMENTOS POR FASE
# ============================================================

proxy_df = dfs["segment_level_proxy_groundtruth"]
role_col_proxy = find_first_col(proxy_df, ["official_role_proxy", "proxy_role", "probable_role", "role_proxy"]) if len(proxy_df) else None
n_proxy_segments = int(proxy_df[role_col_proxy].notna().sum()) if len(proxy_df) and role_col_proxy else np.nan

stage_rows = [
    {"stage": "Segmentos regulares", "value": len(dfs["regular_segments"]) if len(dfs["regular_segments"]) else np.nan},
    {"stage": "Segmentos puntuados", "value": len(dfs["scored_segments"]) if len(dfs["scored_segments"]) else np.nan},
    {"stage": "Segmentos válidos", "value": len(dfs["valid_segments"]) if len(dfs["valid_segments"]) else np.nan},
    {"stage": "Segmentos finales post-relabeling", "value": len(dfs["final_segments"]) if len(dfs["final_segments"]) else np.nan},
    {"stage": "Segmentos finales fusionados", "value": len(dfs["final_merged_segments"]) if len(dfs["final_merged_segments"]) else np.nan},
    {"stage": "Segmentos transcritos", "value": len(dfs["transcribed_segments"]) if len(dfs["transcribed_segments"]) else np.nan},
    {"stage": "Segmentos con proxy textual", "value": n_proxy_segments},
]

funnel_df = plot_segment_funnel(stage_rows, filename="01_pipeline_segment_funnel.png")
save_table(funnel_df, "03_pipeline_segment_funnel.csv")
display(funnel_df)

## 2. Fase 00 — Inventario y limpieza acústica

Esta sección aprovecha los outputs de EDA/limpieza si están disponibles. Si todavía no están en la ruta esperada, el notebook continúa con las siguientes fases.

In [ ]:
# ============================================================
# CELDA 8 - VISUALIZACIONES DE INVENTARIO Y LIMPIEZA
# ============================================================

# Elegir el inventario disponible
inventory_df = pd.DataFrame()
for name in ["audio_inventory_integrated", "audio_inventory_private"]:
    if len(dfs[name]):
        inventory_df = dfs[name].copy()
        print("Inventario usado:", name)
        break

if inventory_df.empty:
    print("No se encontró inventario de EDA. Se omite esta fase visual.")
else:
    display(inventory_df.head())

    # Distribución de duración si existe alguna columna compatible
    duration_col = find_first_col(inventory_df, [
        "duration_sec", "duration_seconds", "raw_duration_sec", "audio_duration_sec", "duration"
    ])
    if duration_col:
        plot_hist(
            inventory_df[duration_col] / 60,
            "Distribución de duración de audios",
            "Duración (minutos)",
            "02_audio_duration_distribution.png",
            bins=30,
            clip_quantile=0.99,
        )

    # Comparativa bruto vs limpio si ambas columnas existen
    raw_duration_col = find_first_col(inventory_df, ["raw_duration_sec", "duration_sec_raw", "original_duration_sec"])
    clean_duration_col = find_first_col(inventory_df, ["clean_duration_sec", "duration_sec_clean", "non_silent_duration_sec"])

    if raw_duration_col and clean_duration_col:
        total_raw_h = inventory_df[raw_duration_col].sum() / 3600
        total_clean_h = inventory_df[clean_duration_col].sum() / 3600
        duration_compare = pd.DataFrame({
            "stage": ["Corpus bruto", "Corpus limpio"],
            "duration_hours": [total_raw_h, total_clean_h],
        })
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.bar(duration_compare["stage"], duration_compare["duration_hours"], color=PALETTE[:2])
        ax.set_title("Duración total antes y después de la limpieza")
        ax.set_ylabel("Horas")
        for i, v in enumerate(duration_compare["duration_hours"]):
            ax.text(i, v, f"{v:.2f} h", ha="center", va="bottom")
        save_fig("03_raw_vs_clean_duration_hours.png")
        save_table(duration_compare, "04_raw_vs_clean_duration_hours.csv")
        display(duration_compare)

## 3. Fase 01 — Diarización, anchors y reetiquetado

Estas figuras resumen la transformación acústica del corpus: segmentos generados, filtros de calidad, selección de anchors y reetiquetado basado en embeddings.

In [ ]:
# ============================================================
# CELDA 9 - DIARIZACIÓN: SEGMENTOS POR AUDIO Y RATIO DE VALIDEZ
# ============================================================

df_summary = dfs["diarization_summary"].copy()

if df_summary.empty:
    print("No se encontró diarization_summary.csv")
else:
    audio_col = detect_audio_col(df_summary)
    print("Audios en summary:", len(df_summary))
    display(df_summary.head())

    # Segmentos por audio
    seg_count_col = find_first_col(df_summary, ["n_scored_segments", "n_regular_segments", "n_segments"])
    if seg_count_col:
        plot_hist(
            df_summary[seg_count_col],
            "Distribución de segmentos por audio",
            "Segmentos por audio",
            "04_segments_per_audio_distribution.png",
            bins=30,
            clip_quantile=0.99,
        )

    # Ratio valid/scored
    if {"n_valid_segments", "n_scored_segments"}.issubset(df_summary.columns):
        df_tmp = df_summary.copy()
        df_tmp["valid_ratio"] = np.where(
            df_tmp["n_scored_segments"] > 0,
            df_tmp["n_valid_segments"] / df_tmp["n_scored_segments"],
            np.nan,
        )
        plot_hist(
            df_tmp["valid_ratio"],
            "Ratio de segmentos válidos sobre segmentos puntuados",
            "Ratio valid/scored",
            "05_valid_scored_ratio_distribution.png",
            bins=30,
        )
        valid_ratio_summary = df_tmp["valid_ratio"].describe().reset_index()
        valid_ratio_summary.columns = ["metric", "value"]
        save_table(valid_ratio_summary, "05_valid_scored_ratio_summary.csv")
        display(valid_ratio_summary)

In [ ]:
# ============================================================
# CELDA 10 - ANCHORS Y SOLAPAMIENTO
# ============================================================

df_anchors = dfs["anchor_segments"].copy()
df_scored = dfs["scored_segments"].copy()

if len(df_anchors):
    audio_col = detect_audio_col(df_anchors)
    speaker_col = find_first_col(df_anchors, ["speaker", "speaker_final", "speaker_original", "label"])

    if audio_col:
        anchors_per_audio = df_anchors.groupby(audio_col).size().sort_values(ascending=False)
        plot_hist(
            anchors_per_audio,
            "Anchors seleccionados por audio",
            "Anchors por audio",
            "06_anchors_per_audio_distribution.png",
            bins=30,
        )
        save_table(anchors_per_audio.reset_index(name="n_anchors"), "06_anchors_per_audio.csv")

    if speaker_col:
        anchor_speaker_counts = df_anchors[speaker_col].value_counts()
        plot_bar_counts(
            anchor_speaker_counts,
            "Anchors seleccionados por speaker",
            "Speaker",
            "Número de anchors",
            "07_anchor_counts_by_speaker.png",
            horizontal=False,
        )
        save_table(anchor_speaker_counts.reset_index().rename(columns={"index": speaker_col, speaker_col: "n_anchors"}), "07_anchor_counts_by_speaker.csv")
else:
    print("No se encontraron anchors globales.")

# Solapamiento global en segmentos puntuados
if len(df_scored) and "overlap_ratio" in df_scored.columns:
    overlap_summary = pd.DataFrame({
        "metric": [
            "mean_overlap_ratio",
            "median_overlap_ratio",
            "max_overlap_ratio",
            "segments_without_overlap",
            "segments_overlap_gt_0_20",
            "share_without_overlap",
            "share_overlap_gt_0_20",
        ],
        "value": [
            df_scored["overlap_ratio"].mean(),
            df_scored["overlap_ratio"].median(),
            df_scored["overlap_ratio"].max(),
            int((df_scored["overlap_ratio"] == 0).sum()),
            int((df_scored["overlap_ratio"] > 0.20).sum()),
            float((df_scored["overlap_ratio"] == 0).mean()),
            float((df_scored["overlap_ratio"] > 0.20).mean()),
        ]
    })
    save_table(overlap_summary, "08_overlap_summary_scored_segments.csv")
    display(overlap_summary)

    plot_hist(
        df_scored["overlap_ratio"],
        "Distribución del overlap ratio en segmentos puntuados",
        "Overlap ratio",
        "08_overlap_ratio_distribution.png",
        bins=30,
    )
else:
    print("No se encontró overlap_ratio en segmentos puntuados.")

In [ ]:
# ============================================================
# CELDA 11 - REETIQUETADO: CAMBIOS Y MATRIZ ORIGINAL VS FINAL
# ============================================================

df_final = dfs["final_segments"].copy()
df_changed = dfs["changed_segments"].copy()

if df_final.empty:
    print("No se encontró all_final_segments.csv")
else:
    n_final = len(df_final)
    n_changed = len(df_changed) if len(df_changed) else np.nan

    # Si no existe changed_segments, calcular por columnas original/final
    if pd.isna(n_changed) and {"speaker_original", "speaker_final"}.issubset(df_final.columns):
        n_changed = int((df_final["speaker_original"] != df_final["speaker_final"]).sum())

    if not pd.isna(n_changed):
        relabel_counts = pd.Series({
            "Sin cambio": int(n_final - n_changed),
            "Reetiquetados": int(n_changed),
        })
        plot_bar_counts(
            relabel_counts,
            "Segmentos finales según efecto del reetiquetado",
            "Categoría",
            "Número de segmentos",
            "09_relabelled_vs_unchanged_segments.png",
        )
        save_table(relabel_counts.reset_index().rename(columns={"index": "category", 0: "n_segments"}), "09_relabelled_vs_unchanged_segments.csv")
        print("Porcentaje reetiquetado:", round(n_changed / n_final * 100, 2), "%")

    # Matriz speaker_original vs speaker_final
    if {"speaker_original", "speaker_final"}.issubset(df_final.columns):
        cm = pd.crosstab(df_final["speaker_original"], df_final["speaker_final"])
        save_table(cm.reset_index(), "10_speaker_original_vs_final_matrix.csv")
        display(cm)
        plot_crosstab_heatmap(
            cm,
            "Matriz de consistencia: speaker original vs speaker final",
            "10_speaker_original_vs_final_matrix.png",
        )

    # Margen de distancia
    margin_col = find_first_col(df_final, ["distance_margin", "margin"])
    if margin_col:
        plot_hist(
            df_final[margin_col],
            "Distribución del margen de distancia en el reetiquetado",
            "Margen de distancia",
            "11_distance_margin_distribution.png",
            bins=30,
            clip_quantile=0.99,
        )

## 4. Fase 02 — Sensibilidad de parámetros internos

Esta sección lee los resultados del notebook de validación interna, cuando existan. Sirve para elegir figuras o tablas de apoyo sobre sensibilidad del margen de reetiquetado y criterios de anchors.

In [ ]:
# ============================================================
# CELDA 12 - SENSIBILIDAD DEL MARGEN DE RELABELING
# ============================================================

df_margin = dfs["relabel_margin_sensitivity"].copy()

# Fallback: buscar cualquier CSV de sensibilidad dentro de la carpeta
if df_margin.empty:
    candidates = sorted((DIARIZATION_DIR / "relabel_margin_sensitivity").glob("*.csv"))
    for p in candidates:
        tmp = read_csv_optional(p)
        if len(tmp) and any("margin" in c.lower() for c in tmp.columns):
            df_margin = tmp.copy()
            print("Sensibilidad de margen leída desde:", p)
            break

if df_margin.empty:
    print("No se encontró tabla de sensibilidad de RELABEL_MIN_MARGIN.")
else:
    display(df_margin.head())

    margin_col = find_first_col(df_margin, ["margin", "relabel_margin", "RELABEL_MIN_MARGIN"])
    changed_col = find_first_col(df_margin, ["n_changed", "n_relabelled", "changed_segments", "n_changed_segments"])
    ratio_col = find_first_col(df_margin, ["changed_ratio", "relabel_ratio", "pct_relabelled", "changed_pct"])

    if margin_col and changed_col:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(df_margin[margin_col], df_margin[changed_col], marker="o", color=PALETTE[0])
        ax.set_title("Sensibilidad del reetiquetado según margen mínimo")
        ax.set_xlabel("Margen mínimo de reetiquetado")
        ax.set_ylabel("Segmentos reetiquetados")
        save_fig("12_relabel_margin_sensitivity_changed_segments.png")

    if margin_col and ratio_col:
        fig, ax = plt.subplots(figsize=(8, 5))
        y = pd.to_numeric(df_margin[ratio_col], errors="coerce")
        if y.max() <= 1.5:
            y = y * 100
        ax.plot(df_margin[margin_col], y, marker="o", color=PALETTE[1])
        ax.set_title("Porcentaje reetiquetado según margen mínimo")
        ax.set_xlabel("Margen mínimo de reetiquetado")
        ax.set_ylabel("Segmentos reetiquetados (%)")
        save_fig("13_relabel_margin_sensitivity_percentage.png")

    save_table(df_margin, "12_relabel_margin_sensitivity.csv")

## 5. Fase 04 — Consolidación de segmentos

Esta sección resume la base granular final que alimenta la transcripción y las fases textuales.

In [ ]:
# ============================================================
# CELDA 13 - CONSOLIDACIÓN DE SEGMENTOS FINALES
# ============================================================

df_merged = dfs["final_merged_segments"].copy()

if df_merged.empty:
    print("No se encontró consolidado all_final_merged_segments.csv")
else:
    audio_col = detect_audio_col(df_merged)
    speaker_col = find_first_col(df_merged, ["speaker_final", "speaker", "label"])
    start_col, end_col = detect_start_end_cols(df_merged)

    print("Segmentos fusionados:", len(df_merged))
    if audio_col:
        print("Audios en consolidado:", df_merged[audio_col].nunique())
        segs_per_audio = df_merged.groupby(audio_col).size()
        plot_hist(
            segs_per_audio,
            "Segmentos finales fusionados por audio",
            "Segmentos por audio",
            "14_merged_segments_per_audio.png",
            bins=30,
            clip_quantile=0.99,
        )
        save_table(segs_per_audio.reset_index(name="n_merged_segments"), "14_merged_segments_per_audio.csv")

    if speaker_col:
        speaker_counts = df_merged[speaker_col].value_counts()
        plot_bar_counts(
            speaker_counts,
            "Distribución de segmentos finales por speaker",
            "Speaker",
            "Número de segmentos",
            "15_merged_segments_by_speaker.png",
        )

    duration_col = find_first_col(df_merged, ["duration", "duration_sec", "segment_duration"])
    if duration_col:
        plot_hist(
            df_merged[duration_col],
            "Distribución de duración de segmentos fusionados",
            "Duración del segmento (s)",
            "16_merged_segment_duration_distribution.png",
            bins=40,
            clip_quantile=0.99,
        )

## 6. Fase 05 — Transcripción automática segmentada

Esta sección resume la cobertura de Whisper a nivel de segmento, manteniendo timestamps y speaker final.

In [ ]:
# ============================================================
# CELDA 14 - COBERTURA DE TRANSCRIPCIÓN WHISPER
# ============================================================

df_trans = dfs["transcribed_segments"].copy()

if df_trans.empty:
    print("No se encontró salida global de transcripción segmentada.")
else:
    audio_col = detect_audio_col(df_trans)
    text_col = detect_text_col(df_trans)
    status_col = find_first_col(df_trans, ["transcription_status", "status", "transcription_state"])
    start_col, end_col = detect_start_end_cols(df_trans)

    print("Segmentos transcritos:", len(df_trans))
    if audio_col:
        print("Audios transcritos:", df_trans[audio_col].nunique())
    print("Columna texto:", text_col)
    print("Columna estado:", status_col)

    if text_col:
        df_trans["_word_count"] = compute_word_count_from_text(df_trans, text_col)
        df_trans["_has_text"] = df_trans[text_col].fillna("").astype(str).str.strip().ne("")

        text_counts = pd.Series({
            "Con texto": int(df_trans["_has_text"].sum()),
            "Sin texto": int((~df_trans["_has_text"]).sum()),
        })
        plot_bar_counts(
            text_counts,
            "Cobertura de transcripción segmentada",
            "Estado",
            "Número de segmentos",
            "17_transcription_text_coverage.png",
        )
        save_table(text_counts.reset_index().rename(columns={"index": "status", 0: "n_segments"}), "17_transcription_text_coverage.csv")

        plot_hist(
            df_trans.loc[df_trans["_has_text"], "_word_count"],
            "Distribución de longitud textual por segmento",
            "Palabras por segmento",
            "18_transcription_word_count_distribution.png",
            bins=40,
            clip_quantile=0.99,
        )

    if status_col:
        status_counts = df_trans[status_col].value_counts(dropna=False)
        plot_bar_counts(
            status_counts,
            "Estados de transcripción por segmento",
            "Estado",
            "Número de segmentos",
            "19_transcription_status_counts.png",
            horizontal=True,
        )
        save_table(status_counts.reset_index().rename(columns={"index": status_col, status_col: "n_segments"}), "19_transcription_status_counts.csv")

    # Muestra compacta con timestamps para memoria/auditoría
    if text_col:
        df_examples = add_interval_column(df_trans, start_col, end_col)
        cols = [c for c in [audio_col, "interval", "speaker_final", "speaker", status_col, "_word_count", text_col] if c and c in df_examples.columns]
        examples = df_examples[df_examples[text_col].fillna("").astype(str).str.strip().ne("")][cols].head(10)
        save_table(examples, "20_transcription_examples_with_timestamps.csv")
        display(examples)

## 7. Fase 06 — Metadata oficial, alineación textual y proxy ground truth

Esta sección resume la construcción del proxy textual que permite aproximar la asignación funcional de speakers a roles cliente/agente sin anotación manual completa.

In [ ]:
# ============================================================
# CELDA 15 - METADATA OFICIAL Y TURNOS EXTRAÍDOS
# ============================================================

df_meta_audit = dfs["metadata_join_audit_by_audio"].copy()
df_turns = dfs["official_transcription_turns_extracted"].copy()
df_role_presence = dfs["official_role_presence_by_audio"].copy()
df_role_summary = dfs["metadata_role_summary"].copy()

if len(df_meta_audit):
    print("Audios en auditoría de metadata:", len(df_meta_audit))
    display(df_meta_audit.head())

    # Si existe alguna columna booleana de match/cobertura, graficarla
    possible_cols = [c for c in df_meta_audit.columns if any(k in c.lower() for k in ["match", "found", "has_", "with_"])]
    bool_cols = []
    for c in possible_cols:
        vals = set(df_meta_audit[c].dropna().astype(str).str.lower().unique())
        if vals.issubset({"true", "false", "1", "0", "yes", "no", "si", "sí"}) or df_meta_audit[c].dropna().isin([0, 1, True, False]).all():
            bool_cols.append(c)

    if bool_cols:
        coverage_rows = []
        for c in bool_cols[:8]:
            s = df_meta_audit[c].astype(str).str.lower().isin(["true", "1", "yes", "si", "sí"])
            coverage_rows.append({"check": c, "n_true": int(s.sum()), "share_true": float(s.mean())})
        coverage_df = pd.DataFrame(coverage_rows)
        save_table(coverage_df, "21_metadata_coverage_checks.csv")
        display(coverage_df)

if len(df_turns):
    print("Turnos oficiales extraídos:", len(df_turns))
    role_col = find_first_col(df_turns, ["official_role", "role", "speaker_role"])
    if role_col:
        turn_role_counts = df_turns[role_col].value_counts()
        plot_bar_counts(
            turn_role_counts,
            "Turnos oficiales extraídos por rol",
            "Rol oficial",
            "Número de turnos",
            "21_official_turns_by_role.png",
        )
        save_table(turn_role_counts.reset_index().rename(columns={"index": role_col, role_col: "n_turns"}), "22_official_turns_by_role.csv")

    text_col = detect_text_col(df_turns)
    if text_col:
        df_turns["_word_count"] = compute_word_count_from_text(df_turns, text_col)
        plot_hist(
            df_turns["_word_count"],
            "Longitud de turnos en transcripción oficial",
            "Palabras por turno oficial",
            "22_official_turn_word_count_distribution.png",
            bins=40,
            clip_quantile=0.99,
        )
else:
    print("No se encontró official_transcription_turns_extracted.csv")

if len(df_role_summary):
    print("Resumen de roles oficiales:")
    display(df_role_summary)
    save_table(df_role_summary, "23_metadata_role_summary.csv")

In [ ]:
# ============================================================
# CELDA 16 - ALINEACIÓN TEXTUAL: CANDIDATOS, MATCHES Y SENSIBILIDAD
# ============================================================

df_candidates = dfs["text_alignment_candidates"].copy()
df_matches = dfs["text_alignment_matches"].copy()
df_threshold = dfs["text_alignment_threshold_sensitivity"].copy()

# Barras candidatos vs matches aceptados
alignment_counts = pd.Series({
    "Candidatos": int(len(df_candidates)) if len(df_candidates) else 0,
    "Matches aceptados": int(len(df_matches)) if len(df_matches) else 0,
})

if alignment_counts.sum() > 0:
    plot_bar_counts(
        alignment_counts,
        "Alineación textual oficial ↔ Whisper",
        "Tipo",
        "Número de pares",
        "23_text_alignment_candidates_vs_matches.png",
    )
    save_table(alignment_counts.reset_index().rename(columns={"index": "type", 0: "n_pairs"}), "24_text_alignment_candidates_vs_matches.csv")
else:
    print("No se encontraron candidatos ni matches de alineación textual.")

# Distribución de score combinado
if len(df_candidates):
    score_col = find_first_col(df_candidates, ["combined_score", "char_cosine", "token_containment", "token_jaccard"])
    if score_col:
        plot_hist(
            df_candidates[score_col],
            f"Distribución de score de alineación ({score_col})",
            score_col,
            "24_text_alignment_score_distribution_candidates.png",
            bins=30,
        )

# Matches por rol oficial
if len(df_matches):
    role_col = find_first_col(df_matches, ["official_role", "role"])
    if role_col:
        match_role_counts = df_matches[role_col].value_counts()
        plot_bar_counts(
            match_role_counts,
            "Matches textuales aceptados por rol oficial",
            "Rol oficial",
            "Número de matches",
            "25_text_alignment_matches_by_role.png",
        )
        save_table(match_role_counts.reset_index().rename(columns={"index": role_col, role_col: "n_matches"}), "25_text_alignment_matches_by_role.csv")

# Sensibilidad de umbrales
if len(df_threshold):
    display(df_threshold.head())
    if {"metric_used", "threshold", "accepted_matches"}.issubset(df_threshold.columns):
        fig, ax = plt.subplots(figsize=(9, 5))
        for metric, df_m in df_threshold.groupby("metric_used"):
            df_m = df_m.sort_values("threshold")
            ax.plot(df_m["threshold"], df_m["accepted_matches"], marker="o", label=metric)
        ax.set_title("Sensibilidad de matches aceptados por umbral")
        ax.set_xlabel("Umbral")
        ax.set_ylabel("Matches aceptados")
        ax.legend()
        save_fig("26_text_alignment_threshold_sensitivity_matches.png")
    save_table(df_threshold, "26_text_alignment_threshold_sensitivity.csv")
else:
    print("No se encontró text_alignment_threshold_sensitivity.csv")

In [ ]:
# ============================================================
# CELDA 17 - MAPPING SPEAKER → ROL Y PROXY A NIVEL SEGMENTO
# ============================================================

df_mapping = dfs["speaker_role_mapping_textual"].copy()
df_proxy = dfs["segment_level_proxy_groundtruth"].copy()

if len(df_mapping):
    print("Mappings speaker→rol:", len(df_mapping))
    display(df_mapping.head())

    status_col = find_first_col(df_mapping, ["role_mapping_status", "mapping_status", "status"])
    role_col = find_first_col(df_mapping, ["probable_role", "official_role_proxy", "proxy_role", "role"])
    conf_col = find_first_col(df_mapping, ["role_confidence", "proxy_confidence", "confidence"])

    if status_col:
        mapping_status_counts = df_mapping[status_col].value_counts(dropna=False)
        plot_bar_counts(
            mapping_status_counts,
            "Estado del mapping speaker→rol",
            "Estado",
            "Número de mappings",
            "27_speaker_role_mapping_status.png",
            horizontal=True,
        )

    if role_col:
        mapping_role_counts = df_mapping[role_col].value_counts(dropna=False)
        plot_bar_counts(
            mapping_role_counts,
            "Roles inferidos por mapping textual",
            "Rol inferido",
            "Número de mappings",
            "28_speaker_role_mapping_by_role.png",
        )

    if conf_col:
        plot_hist(
            df_mapping[conf_col],
            "Distribución de confianza del mapping speaker→rol",
            "Confianza",
            "29_speaker_role_mapping_confidence.png",
            bins=20,
        )
else:
    print("No se encontró speaker_role_mapping_textual.csv")

if len(df_proxy):
    print("Segmentos en salida proxy:", len(df_proxy))
    role_col = find_first_col(df_proxy, ["official_role_proxy", "proxy_role", "probable_role", "role_proxy"])
    method_col = find_first_col(df_proxy, ["proxy_method", "method"])
    conf_col = find_first_col(df_proxy, ["proxy_confidence", "role_confidence", "confidence"])
    audio_col = detect_audio_col(df_proxy)

    if role_col:
        n_labeled = int(df_proxy[role_col].notna().sum())
        n_unlabeled = int(len(df_proxy) - n_labeled)
        proxy_coverage = pd.Series({
            "Con proxy textual": n_labeled,
            "Sin proxy textual": n_unlabeled,
        })
        plot_bar_counts(
            proxy_coverage,
            "Cobertura del proxy textual a nivel segmento",
            "Estado",
            "Número de segmentos",
            "30_segment_level_proxy_coverage.png",
        )
        save_table(proxy_coverage.reset_index().rename(columns={"index": "status", 0: "n_segments"}), "30_segment_level_proxy_coverage.csv")

        role_counts = df_proxy[role_col].dropna().value_counts()
        if len(role_counts):
            plot_bar_counts(
                role_counts,
                "Distribución de roles propagados por proxy textual",
                "Rol proxy",
                "Número de segmentos",
                "31_segment_level_proxy_roles.png",
            )
            save_table(role_counts.reset_index().rename(columns={"index": role_col, role_col: "n_segments"}), "31_segment_level_proxy_roles.csv")

        if audio_col:
            audios_labeled = df_proxy.loc[df_proxy[role_col].notna(), audio_col].nunique()
            audios_total = df_proxy[audio_col].nunique()
            audio_coverage = pd.DataFrame({
                "metric": ["Audios totales", "Audios con proxy textual"],
                "value": [audios_total, audios_labeled],
            })
            save_table(audio_coverage, "32_proxy_audio_coverage.csv")
            display(audio_coverage)

    if method_col:
        method_counts = df_proxy[method_col].value_counts(dropna=False)
        plot_bar_counts(
            method_counts,
            "Método de asignación del proxy textual",
            "Método",
            "Número de segmentos",
            "32_proxy_method_counts.png",
            horizontal=True,
        )

    if conf_col:
        plot_hist(
            df_proxy[conf_col],
            "Distribución de confianza del proxy textual por segmento",
            "Confianza proxy",
            "33_segment_level_proxy_confidence.png",
            bins=20,
        )
else:
    print("No se encontró segment_level_proxy_groundtruth.csv")

In [ ]:
# ============================================================
# CELDA 18 - MÉTRICAS HOLDOUT / EVALUACIÓN INTERNA DEL PROXY
# ============================================================

df_holdout_metrics = dfs["holdout_role_mapping_metrics"].copy()
df_holdout_pred = dfs["holdout_role_mapping_predictions"].copy()

if len(df_holdout_metrics):
    print("Métricas holdout del proxy:")
    display(df_holdout_metrics)
    save_table(df_holdout_metrics, "33_holdout_role_mapping_metrics.csv")

    metric_col = find_first_col(df_holdout_metrics, ["metric"])
    value_col = find_first_col(df_holdout_metrics, ["value"])
    if metric_col and value_col:
        df_plot = df_holdout_metrics.copy()
        df_plot[value_col] = pd.to_numeric(df_plot[value_col], errors="coerce")
        df_plot = df_plot.dropna(subset=[value_col])
        # Mostrar solo métricas principales entre 0 y 1 o conteos pequeños razonables
        df_plot = df_plot[df_plot[metric_col].astype(str).str.contains("precision|recall|f1", case=False, regex=True)]
        if len(df_plot):
            fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(df_plot))))
            ax.barh(df_plot[metric_col], df_plot[value_col], color=PALETTE[0])
            ax.set_title("Métricas internas holdout del proxy textual")
            ax.set_xlabel("Valor")
            ax.set_ylabel("Métrica")
            ax.invert_yaxis()
            for i, v in enumerate(df_plot[value_col]):
                ax.text(v, i, f" {v:.2f}", va="center")
            save_fig("34_holdout_proxy_metrics.png")
else:
    print("No se encontró holdout_role_mapping_metrics.csv")

if len(df_holdout_pred):
    role_true_col = find_first_col(df_holdout_pred, ["official_role", "true_role"])
    role_pred_col = find_first_col(df_holdout_pred, ["predicted_role_from_train", "predicted_role"])
    if role_true_col and role_pred_col:
        ct = pd.crosstab(df_holdout_pred[role_true_col], df_holdout_pred[role_pred_col])
        save_table(ct.reset_index(), "34_holdout_proxy_confusion_matrix.csv")
        plot_crosstab_heatmap(
            ct,
            "Matriz holdout: rol oficial vs rol predicho",
            "35_holdout_proxy_confusion_matrix.png",
        )

## 8. Inventario final de tablas y figuras

Esta celda lista los archivos generados por el Notebook 11 para que puedas llevarlos directamente a la memoria o a la presentación.

In [ ]:
# ============================================================
# CELDA 19 - INVENTARIO FINAL DE OUTPUTS GENERADOS
# ============================================================

fig_files = sorted(FIGURES_DIR.glob("*.png"))
table_files = sorted(TABLES_DIR.glob("*.csv"))

outputs_inventory = pd.DataFrame(
    [{"type": "figure", "filename": p.name, "path": str(p)} for p in fig_files] +
    [{"type": "table", "filename": p.name, "path": str(p)} for p in table_files]
)

print("Figuras generadas:", len(fig_files))
print("Tablas generadas:", len(table_files))

display(outputs_inventory)
outputs_inventory.to_csv(NOTEBOOK11_OUTPUT_DIR / "notebook11_outputs_inventory.csv", index=False)
print("Inventario guardado en:", NOTEBOOK11_OUTPUT_DIR / "notebook11_outputs_inventory.csv")

## Notas de uso

- Este notebook debe ejecutarse después de los notebooks 00 a 06.
- Si una fase aún no se ha ejecutado, sus visualizaciones se omiten sin interrumpir el resto del notebook.
- Las figuras generadas se guardan en `data/global_visualizations_memory_presentation/figures`.
- Las tablas de apoyo se guardan en `data/global_visualizations_memory_presentation/tables`.
- El notebook puede ampliarse después para incorporar sentimiento, palabras clave y huella de voz, manteniendo la misma estructura por fases.